Cell 1 — Đọc dữ liệu


In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/interim/flight_data_7m_cleaned.csv")

df = pd.read_csv(DATA_PATH)

print(f"Kích thước dữ liệu: {df.shape}")

Kích thước dữ liệu: (6965266, 35)


Xác định target

In [2]:
TARGET_COLUMNS = [
    "carrier_delay",
    "weather_delay",
    "nas_delay",
    "security_delay",
    "late_aircraft_delay"
]

Correlation giữa biến số và 5 nguyên nhân delay

In [3]:
NUMERIC_COLUMNS = df.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

FEATURE_NUMERIC_COLUMNS = [
    col for col in NUMERIC_COLUMNS
    if col not in TARGET_COLUMNS
]

correlation_with_targets = (
    df[FEATURE_NUMERIC_COLUMNS + TARGET_COLUMNS]
    .corr(method="pearson")
    .loc[FEATURE_NUMERIC_COLUMNS, TARGET_COLUMNS]
)

print(correlation_with_targets)

                     carrier_delay  weather_delay  nas_delay  security_delay  \
year                           NaN            NaN        NaN             NaN   
month                    -0.007274      -0.009908  -0.020818       -0.004303   
day_of_month              0.004180       0.001907   0.000528        0.000174   
day_of_week               0.008507      -0.002725   0.005625        0.001092   
op_carrier_fl_num         0.000302       0.017885  -0.011820       -0.002534   
crs_dep_time              0.020500       0.015534   0.036737        0.006844   
dep_time                  0.027583       0.011939   0.057543        0.008084   
dep_delay                 0.695595       0.297035   0.250288        0.030665   
taxi_out                  0.036814       0.054682   0.314926        0.002655   
wheels_off                0.023901       0.010434   0.062730        0.007921   
wheels_on                -0.011289      -0.002485   0.037952        0.003374   
taxi_in                   0.016763      

Tìm các feature tương quan cao với nhau

In [4]:
feature_corr = df[FEATURE_NUMERIC_COLUMNS].corr(method="pearson")

high_corr_pairs = []

for i in range(len(feature_corr.columns)):
    for j in range(i + 1, len(feature_corr.columns)):
        col1 = feature_corr.columns[i]
        col2 = feature_corr.columns[j]
        corr = feature_corr.iloc[i, j]

        if abs(corr) >= 0.8:
            high_corr_pairs.append({
                "feature_1": col1,
                "feature_2": col2,
                "correlation": corr
            })

high_corr_pairs_df = pd.DataFrame(high_corr_pairs)

print(high_corr_pairs_df)

              feature_1            feature_2  correlation
0          crs_dep_time             dep_time     0.944776
1          crs_dep_time           wheels_off     0.909840
2              dep_time           wheels_off     0.963160
3             dep_delay            arr_delay     0.970694
4             wheels_on         crs_arr_time     0.829728
5             wheels_on             arr_time     0.956983
6          crs_arr_time             arr_time     0.816062
7      crs_elapsed_time  actual_elapsed_time     0.981425
8      crs_elapsed_time             air_time     0.989446
9      crs_elapsed_time             distance     0.983404
10  actual_elapsed_time             air_time     0.986430
11  actual_elapsed_time             distance     0.968641
12             air_time             distance     0.984400


Association Analysis cho biến categorical

In [5]:
CATEGORICAL_COLUMNS = [
    "op_unique_carrier",
    "origin",
    "origin_city_name",
    "origin_state_nm",
    "dest",
    "dest_city_name",
    "dest_state_nm"
]
for target in TARGET_COLUMNS:
    df[f"{target}_present"] = (
        df[target] > 0
    ).astype(int)

Cramér's V

In [6]:
import numpy as np
from scipy.stats import chi2_contingency


def cramers_v(x, y):
    table = pd.crosstab(x, y)

    chi2 = chi2_contingency(table)[0]

    n = table.sum().sum()
    phi2 = chi2 / n

    r, k = table.shape

    phi2corr = max(
        0,
        phi2 - ((k - 1) * (r - 1)) / (n - 1)
    )

    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)

    denominator = min(kcorr - 1, rcorr - 1)

    if denominator <= 0:
        return 0

    return np.sqrt(phi2corr / denominator)

In [7]:
association_results = []

for feature in CATEGORICAL_COLUMNS:
    for target in TARGET_COLUMNS:

        target_present = f"{target}_present"

        value = cramers_v(
            df[feature],
            df[target_present]
        )

        association_results.append({
            "feature": feature,
            "target": target,
            "cramers_v": value
        })

association_df = pd.DataFrame(association_results)

print(association_df)

              feature               target  cramers_v
0   op_unique_carrier        carrier_delay   0.080387
1   op_unique_carrier        weather_delay   0.069788
2   op_unique_carrier            nas_delay   0.087103
3   op_unique_carrier       security_delay   0.031836
4   op_unique_carrier  late_aircraft_delay   0.132307
5              origin        carrier_delay   0.083012
6              origin        weather_delay   0.072852
7              origin            nas_delay   0.071940
8              origin       security_delay   0.061036
9              origin  late_aircraft_delay   0.095464
10   origin_city_name        carrier_delay   0.082018
11   origin_city_name        weather_delay   0.072172
12   origin_city_name            nas_delay   0.070288
13   origin_city_name       security_delay   0.060972
14   origin_city_name  late_aircraft_delay   0.093931
15    origin_state_nm        carrier_delay   0.061894
16    origin_state_nm        weather_delay   0.053857
17    origin_state_nm       

Lưu kết quả

In [8]:
from pathlib import Path

RESULTS_PATH = Path("../data/interim/correlation_analysis/7m")

RESULTS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

Lưu Pearson: Numeric Feature → Target

In [9]:
correlation_with_targets.to_csv(
    RESULTS_PATH / "numeric_target_correlation_7m.csv",
    encoding="utf-8-sig"
)

print("Đã lưu: numeric_target_correlation_7m.csv")

Đã lưu: numeric_target_correlation_7m.csv


Lưu Pearson: Feature ↔ Feature

In [10]:
high_corr_pairs_df.to_csv(
    RESULTS_PATH / "high_feature_correlation_7m.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu: high_feature_correlation_7m.csv")

Đã lưu: high_feature_correlation_7m.csv


Lưu Cramér's V: Categorical → Target

In [11]:
association_df.to_csv(
    RESULTS_PATH / "categorical_target_association_7m.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu: categorical_target_association_7m.csv")

Đã lưu: categorical_target_association_7m.csv
